# 🎙️ Real-Time Swahili–English AI Interpretation System
**NM-AIST DSAI Capstone | Author: Fred**

**Pipeline:** `Whisper large-v3` → `NLLB-200-distilled-600M` → `Coqui XTTS-v2`

---
### ⚠️ Before running:
1. `Runtime` → `Change runtime type` → **T4 GPU**
2. Run cells **top to bottom**, one at a time
3. Update `GITHUB_USERNAME` in Cell 1 with your actual GitHub username

## 📥 Cell 1 — Clone Repository from GitHub

In [ ]:
# ── EDIT THIS ────────────────────────────────────────────────────────────────
GITHUB_USERNAME = "claverfred"   # <-- your GitHub username
REPO_NAME       = "swahili-interpreter"
BRANCH          = "main"
# ─────────────────────────────────────────────────────────────────────────────

import os, sys, subprocess

REPO_URL  = f"https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"
REPO_PATH = f"/content/{REPO_NAME}"

if os.path.exists(REPO_PATH):
    print('Repo already exists — pulling latest changes...')
    os.chdir(REPO_PATH)
    result = subprocess.run(['git', 'pull', 'origin', BRANCH],
                             capture_output=True, text=True)
    print(result.stdout or result.stderr)
else:
    print(f'Cloning {REPO_URL}...')
    result = subprocess.run(
        ['git', 'clone', '-b', BRANCH, REPO_URL, REPO_PATH],
        capture_output=True, text=True
    )
    print(result.stdout)
    if result.returncode != 0:
        print('❌ Clone failed:')
        print(result.stderr)
        raise RuntimeError(
            f'Git clone failed. Check that:\n'
            f'  1. The repo exists at {REPO_URL}\n'
            f'  2. It is public (or you have auth)\n'
            f"  3. The branch '{BRANCH}' exists\n\n"
            f'Error: {result.stderr}'
        )
    os.chdir(REPO_PATH)

if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

print(f'\n✅ Working directory: {os.getcwd()}')
print('Repository contents:')
subprocess.run(['find', '.', '-not', '-path', '*/.git/*',
                 '-not', '-path', '*/__pycache__/*'],
                capture_output=False)

## 📦 Cell 2 — Install Dependencies

In [ ]:
import subprocess, sys

print('Installing from requirements.txt...')
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'],
    capture_output=True, text=True
)
if result.returncode != 0:
    print('⚠️  Some packages had issues:')
    print(result.stderr[-2000:])
else:
    print('✅ All dependencies installed.')

# Verify critical imports — auto-install anything still missing
checks = [
    ('torch',           'torch'),
    ('faster_whisper',  'faster_whisper'),
    ('transformers',    'transformers'),
    ('TTS',             'TTS'),
    ('edge_tts',        'edge_tts'),
    ('sacrebleu',       'sacrebleu'),
    ('jiwer',           'jiwer'),
    ('loguru',          'loguru'),
    ('yaml',            'pyyaml'),
]
print('\nDependency check:')
missing = []
for import_name, pip_name in checks:
    try:
        __import__(import_name)
        print(f'  ✓ {import_name}')
    except ImportError:
        print(f'  ✗ {import_name} — MISSING')
        missing.append(pip_name)

if missing:
    print(f'\nInstalling missing packages individually: {missing}')
    fix = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q'] + missing,
        capture_output=True, text=True
    )
    if fix.returncode != 0:
        print('❌ Failed to install missing packages:')
        print(fix.stderr[-2000:])
        raise RuntimeError(
            f'Could not install: {missing}. Re-run this cell, or if the '
            f'error persists, restart the runtime (Runtime → Restart session) '
            f'and run all cells from Cell 1 again.'
        )
    print(f'✅ Installed: {missing}')

## 🔧 Cell 3 — Load Config & Check GPU

In [ ]:
import os
os.chdir(REPO_PATH)   # ensure correct working directory

from src.utils import load_config, print_config_summary

config = load_config()
print_config_summary(config)

# Show GPU details
import torch
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f'\nGPU Details:')
    print(f'  Name      : {gpu.name}')
    print(f'  VRAM      : {gpu.total_memory/1024**3:.1f} GB')
    print(f'  CUDA      : {torch.version.cuda}')
else:
    print('\n⚠️  No GPU detected — performance will be limited.')
    print('Go to Runtime → Change runtime type → T4 GPU')

## 📂 Cell 4 — Upload Your Audio File

In [ ]:
from google.colab import files
import shutil

print('Upload a Swahili audio file (mp3, wav, m4a)...')
uploaded = files.upload()

AUDIO_FILE = None
for filename in uploaded.keys():
    dest = f"data/raw/{filename}"
    os.makedirs('data/raw', exist_ok=True)
    shutil.move(filename, dest)
    AUDIO_FILE = dest
    print(f'✓ Saved: {dest}')

# Alternatively, uncomment to use hotuba.mp3 if already uploaded:
# AUDIO_FILE = "data/raw/hotuba.mp3"

print(f'\nReady to interpret: {AUDIO_FILE}')

## ⚙️ Cell 5 — Initialise & Load Pipeline

In [ ]:
from src.pipeline import SwahiliInterpreter

interpreter = SwahiliInterpreter(config)
interpreter.load_all()
print('\n✅ Pipeline ready.')

## ▶️ Cell 6 — Run Full Pipeline

In [ ]:
results = interpreter.interpret(AUDIO_FILE)

# Show first 5 aligned segments
print('\n' + '='*60)
print('SAMPLE OUTPUT — FIRST 5 SEGMENTS')
print('='*60)
for seg in results.segments[:5]:
    print(f"\n[{seg['start']:.1f}s → {seg['end']:.1f}s]")
    print(f"  SW: {seg['swahili']}")
    print(f"  EN: {seg['english']}")

## 🔊 Cell 7 — Play Audio (Input vs Output)

In [ ]:
from IPython.display import Audio, display

print('▶️  INPUT — Original Swahili Audio:')
display(Audio(AUDIO_FILE))

print('\n▶️  OUTPUT — English Interpretation:')
display(Audio(results.output_audio))

## 📊 Cell 8 — ASR Evaluation on Common Voice (WER)

In [ ]:
from src.eval import Evaluator

evaluator   = Evaluator(config)
asr_results = evaluator.evaluate_asr(
    asr_model = interpreter.asr,
    save_path = 'results/eval/asr_eval.json'
)

print(f"\n✅ ASR Evaluation Complete")
print(f"   WER        : {asr_results['wer']:.4f} ({asr_results['wer']*100:.2f}%)")
print(f"   CER        : {asr_results['cer']:.4f} ({asr_results['cer']*100:.2f}%)")
print(f"   Avg latency: {asr_results['avg_latency_sec']:.3f}s/sample")
print(f"   N samples  : {asr_results['n_samples']}")

## 🌍 Cell 9 — MT Model Comparison (NLLB vs Helsinki)

In [ ]:
from src.mt import HelsinkiTranslator

# Use ASR transcript segments as source texts
sw_texts = [s['swahili'] for s in results.segments]
en_refs  = [s['english'] for s in results.segments]  # NLLB as pseudo-reference

helsinki = HelsinkiTranslator(config)

comparison = evaluator.compare_mt_models(
    nllb_model      = interpreter.mt,
    helsinki_model  = helsinki,
    source_texts    = sw_texts,
    reference_texts = en_refs,
)

import pandas as pd
df = pd.DataFrame([
    {'Model': 'NLLB-200-distilled-600M',
     'BLEU' : comparison['nllb']['bleu'],
     'chrF' : comparison['nllb']['chrf'],
     'Time (s)': comparison['nllb']['elapsed'],
     'Seg/sec': comparison['nllb']['seg_per_sec']},
    {'Model': 'Helsinki-NLP opus-swc-en',
     'BLEU' : comparison['helsinki']['bleu'],
     'chrF' : comparison['helsinki']['chrf'],
     'Time (s)': comparison['helsinki']['elapsed'],
     'Seg/sec': comparison['helsinki']['seg_per_sec']},
])
print('\nMT Model Comparison:')
display(df)
print(f"\nWinner: {comparison['winner'].upper()}")

## 📈 Cell 10 — Visualise All Results

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

fig = plt.figure(figsize=(18, 10))
fig.suptitle(
    'Swahili–English AI Interpretation System — Evaluation Results\n'
    'NM-AIST DSAI Capstone Project',
    fontsize=14, fontweight='bold', y=1.01
)
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

COLORS = {'asr': '#2196F3', 'mt': '#4CAF50', 'tts': '#FF9800',
          'nllb': '#4CAF50', 'helsinki': '#F44336'}

# ── 1. Pipeline latency breakdown ─────────────────────────────────────────────
ax1   = fig.add_subplot(gs[0, 0])
t     = results.timing
names = ['ASR\n(Whisper)', 'MT\n(NLLB-200)', 'TTS\n(XTTS-v2)']
vals  = [t['asr_sec'], t['mt_sec'], t['tts_sec']]
cols  = [COLORS['asr'], COLORS['mt'], COLORS['tts']]
bars  = ax1.bar(names, vals, color=cols, edgecolor='white', linewidth=1.5)
for b, v in zip(bars, vals):
    ax1.text(b.get_x()+b.get_width()/2, b.get_height()+0.5,
             f'{v:.1f}s', ha='center', fontweight='bold', fontsize=10)
ax1.set_title('Pipeline Latency by Stage', fontweight='bold')
ax1.set_ylabel('Seconds')
ax1.grid(axis='y', alpha=0.3)

# ── 2. Stage share pie chart ──────────────────────────────────────────────────
ax2  = fig.add_subplot(gs[0, 1])
total = sum(vals)
ax2.pie([v/total*100 for v in vals], labels=names, colors=cols,
        autopct='%1.1f%%', startangle=90, pctdistance=0.75)
ax2.set_title('Latency Distribution', fontweight='bold')

# ── 3. MT model comparison — BLEU ─────────────────────────────────────────────
ax3    = fig.add_subplot(gs[0, 2])
models = ['NLLB-200\n600M', 'Helsinki-NLP\nopus-swc-en']
bleus  = [comparison['nllb']['bleu'], comparison['helsinki']['bleu']]
chrfs  = [comparison['nllb']['chrf'], comparison['helsinki']['chrf']]
x      = np.arange(len(models))
w      = 0.35
ax3.bar(x - w/2, bleus, w, label='BLEU',  color=COLORS['nllb'],     alpha=0.85)
ax3.bar(x + w/2, chrfs, w, label='chrF',  color=COLORS['helsinki'], alpha=0.85)
ax3.set_xticks(x)
ax3.set_xticklabels(models)
ax3.set_title('MT Model Comparison', fontweight='bold')
ax3.set_ylabel('Score')
ax3.legend()
ax3.grid(axis='y', alpha=0.3)

# ── 4. MT latency comparison ──────────────────────────────────────────────────
ax4   = fig.add_subplot(gs[1, 0])
times = [comparison['nllb']['elapsed'], comparison['helsinki']['elapsed']]
bars4 = ax4.bar(models, times,
                color=[COLORS['nllb'], COLORS['helsinki']],
                edgecolor='white', linewidth=1.5)
for b, v in zip(bars4, times):
    ax4.text(b.get_x()+b.get_width()/2, b.get_height()+0.2,
             f'{v:.1f}s', ha='center', fontweight='bold', fontsize=10)
ax4.set_title('MT Inference Time', fontweight='bold')
ax4.set_ylabel('Seconds')
ax4.grid(axis='y', alpha=0.3)

# ── 5. ASR evaluation summary ─────────────────────────────────────────────────
ax5 = fig.add_subplot(gs[1, 1])
metrics = ['WER', 'CER']
scores  = [asr_results['wer'], asr_results['cer']]
bars5   = ax5.bar(metrics, [s*100 for s in scores],
                  color=COLORS['asr'], edgecolor='white', linewidth=1.5)
for b, s in zip(bars5, scores):
    ax5.text(b.get_x()+b.get_width()/2, b.get_height()+0.3,
             f'{s*100:.2f}%', ha='center', fontweight='bold', fontsize=10)
ax5.set_title(f'ASR Evaluation (N={asr_results["n_samples"]})', fontweight='bold')
ax5.set_ylabel('Error Rate (%)')
ax5.grid(axis='y', alpha=0.3)

# ── 6. Scorecard table ────────────────────────────────────────────────────────
ax6 = fig.add_subplot(gs[1, 2])
ax6.axis('off')
table_data = [
    ['Component', 'Model',              'Metric', 'Score'],
    ['ASR',  'Whisper large-v3',        'WER',    f"{asr_results['wer']*100:.2f}%"],
    ['ASR',  'Whisper large-v3',        'CER',    f"{asr_results['cer']*100:.2f}%"],
    ['MT',   'NLLB-200-600M',           'BLEU',   f"{comparison['nllb']['bleu']:.2f}"],
    ['MT',   'Helsinki-NLP',            'BLEU',   f"{comparison['helsinki']['bleu']:.2f}"],
    ['TTS',  'XTTS-v2 / Edge-TTS',     'RTF',    f"{results.timing['tts_sec']:.2f}s"],
]
tbl = ax6.table(
    cellText=table_data[1:], colLabels=table_data[0],
    cellLoc='center', loc='center',
    bbox=[0, 0, 1, 1]
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
ax6.set_title('Phase 1 Scorecard', fontweight='bold', pad=15)

plt.savefig('results/eval/full_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('\n✅ Chart saved: results/eval/full_results.png')

## 💾 Cell 11 — Download Results & Push to GitHub

In [ ]:
# ── Download results zip ──────────────────────────────────────────────────────
import zipfile
from google.colab import files

zip_path = 'swahili_interpreter_results.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, filenames in os.walk('results'):
        for fn in filenames:
            fp = os.path.join(root, fn)
            zf.write(fp)

print(f'✅ Results zipped: {zip_path}')
files.download(zip_path)

In [ ]:
# ── (Optional) Push results back to GitHub ────────────────────────────────────
# Uncomment and fill in your GitHub token to push results after each run.
# Keep your token private — never commit it to the repo.

# GITHUB_TOKEN = "your_token_here"   # from github.com/settings/tokens
# GITHUB_EMAIL = "your@email.com"
# GITHUB_NAME  = "Your Name"

# os.system(f'git config user.email "{GITHUB_EMAIL}"')
# os.system(f'git config user.name  "{GITHUB_NAME}"')

# # Copy results (only JSONs and charts, not large audio)
# os.system('git add results/**/*.json results/**/*.png results/**/*.csv')
# os.system('git commit -m "Add evaluation results from Colab run"')
# os.system(f'git push https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git {BRANCH}')
# print('✅ Results pushed to GitHub.')

print('Uncomment the lines above and add your GitHub token to push results.')

## 📝 Cell 12 — Thesis Findings
Run this last — copy output directly into your thesis write-up.

In [ ]:
t = results.timing
print(f"""
╔══════════════════════════════════════════════════════════════╗
║   PHASE 1 RESULTS — SWAHILI-ENGLISH INTERPRETATION SYSTEM   ║
╠══════════════════════════════════════════════════════════════╣
║  ASR  (Whisper large-v3)                                     ║
║    WER          : {asr_results['wer']*100:.2f}%                                  ║
║    CER          : {asr_results['cer']*100:.2f}%                                  ║
║    Latency      : {t['asr_sec']:.2f}s                                 ║
║    RTF          : {results.asr_meta['rtf']:.4f}                               ║
║    Lang detect  : {results.asr_meta['language']} @ {results.asr_meta['lang_confidence']:.2f} confidence          ║
║                                                              ║
║  MT (NLLB-200-distilled-600M)                                ║
║    BLEU         : {comparison['nllb']['bleu']:.2f}                                ║
║    chrF         : {comparison['nllb']['chrf']:.2f}                                ║
║    Latency      : {t['mt_sec']:.2f}s                                 ║
║                                                              ║
║  MT Baseline (Helsinki-NLP)                                  ║
║    BLEU         : {comparison['helsinki']['bleu']:.2f}                                ║
║    chrF         : {comparison['helsinki']['chrf']:.2f}                                ║
║    Latency      : {comparison['helsinki']['elapsed']:.2f}s                                ║
║                                                              ║
║  TTS (XTTS-v2 / Edge-TTS)                                   ║
║    Latency      : {t['tts_sec']:.2f}s                                 ║
║                                                              ║
║  PIPELINE TOTAL : {t['total_sec']:.2f}s                                ║
║  ASR BOTTLENECK : {t['asr_sec']/t['total_sec']*100:.1f}% of total time                   ║
╚══════════════════════════════════════════════════════════════╝
""")